[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhiyunli/cpsc5610-labs/blob/main/week4/week4_lab_inclass.ipynb)

# Week 4 Lab (Wednesday): Training Real Networks with PyTorch

This lab goes with Lecture 7. It has **four short exercises**. We'll do each one right after the slides on the same idea.

| Exercise | Topic | Time |
|---|---|---|
| 1 | Train an MLP with mini-batches | 13 min |
| 2 | Evaluate it correctly | 8 min |
| 3 | An image classifier | 17 min |
| 4 | Save, load, and check | 7 min |

**How to use it:** replace every `[YOUR CODE]` with one line of code. Run the cells from top to bottom. Later
exercises reuse names from earlier ones (`device`, `train`, `train_data`, ...).

**First:** choose *Runtime → Change runtime type → T4 GPU*, then run the setup cell below. It downloads the data
while we talk.

## Setup (run this first)

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("torchmetrics") is None:           # Colab doesn't always have it
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchmetrics"], check=True)

import torch
import torch.nn as nn
import torchmetrics
import torchvision
import torchvision.transforms.v2 as T
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("device:", device)

# California housing, prepared exactly as in Lecture 6
housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42)
X_train = torch.FloatTensor(X_train)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (torch.FloatTensor(X_valid) - means) / stds
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
n_features = X_train.shape[1]

# Fashion MNIST, for exercise 3
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])
train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=toTensor)
torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(train_and_valid_data, [55_000, 5_000])
class_names = train_and_valid_data.classes
print("housing:", X_train.shape, "  fashion:", len(train_data), len(valid_data))

---
## Exercise 1: Train an MLP with mini-batches (13 min)

**1a.** Build an MLP with `nn.Sequential`: 8 inputs → 50 units → ReLU → 40 units → ReLU → 1 output.

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    [YOUR CODE]  # first layer and its activation
    [YOUR CODE]  # second layer and its activation
    [YOUR CODE]  # the output layer
).to(device)
print("parameters:", sum(p.numel() for p in model.parameters()))   # should print 2531

**1b.** Wrap the training data in a `TensorDataset`, and make a `DataLoader` with batches of 32, shuffled.

In [ ]:
train_dataset = TensorDataset(X_train, y_train)
[YOUR CODE]  # train_loader = ... (batches of 32, shuffled)
print("batches per epoch:", len(train_loader))                     # should print 363

**1c.** Fill in the five phases of the training loop: forward, loss, backward, step, reset.

In [ ]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            [YOUR CODE]  # forward: y_pred = ...
            [YOUR CODE]  # loss = ...
            total_loss += loss.item()
            [YOUR CODE]  # backward
            [YOUR CODE]  # step
            [YOUR CODE]  # reset the gradients
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {total_loss / len(train_loader):.4f}")

**1d.** Train for 10 epochs. The printed loss (the average over each epoch's batches) should fall to about 0.34. (Small differences between machines are normal.)

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.02)
mse = nn.MSELoss()
train(model, optimizer, mse, train_loader, n_epochs=10)

*If you finish early:* try `lr=0.05`, or a third hidden layer. What happens to the loss?

---
## Exercise 2: Evaluate it correctly (8 min)

**2a.** Finish `evaluate()`: switch the model to evaluation mode, and turn off gradient tracking.

In [ ]:
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=32)

def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    [YOUR CODE]  # evaluation mode
    metrics = []
    [YOUR CODE]  # no gradient tracking
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            metrics.append(metric_fn(model(X_batch), y_batch))
    return aggregate_fn(torch.stack(metrics))

def rmse(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean().sqrt()

print("mean of the batch RMSEs:", evaluate(model, valid_loader, rmse).item())

**2b.** Now compute the RMSE over the whole validation set with TorchMetrics: `update()` for every batch, then `compute()`.

In [ ]:
rmse_metric = torchmetrics.MeanSquaredError(squared=False).to(device)
model.eval()
with torch.no_grad():
    for X_batch, y_batch in valid_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        [YOUR CODE]  # add this batch
[YOUR CODE]  # print the final value

**Question:** the two numbers differ. Which one is the real RMSE, and why is the other one different?
*(Write your answer here.)*

---
## Exercise 3: An image classifier (17 min)

**3a.** Look at the data. Each image is 1 × 28 × 28: one channel (gray), 28 rows, 28 columns.

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, i in zip(axes, range(8)):
    image, label = train_data[i]
    ax.imshow(image.squeeze(), cmap="binary")
    ax.set_title(class_names[label], fontsize=9)
    ax.axis("off")
plt.show()

**3b.** Write the model as a class. In `__init__`, build an `nn.Sequential` with `nn.Flatten()`, then
784 → 300 → ReLU → 100 → ReLU → 10. In `forward`, run it. **No softmax at the end.**

In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            [YOUR CODE]  # [batch, 1, 28, 28] -> [batch, 784]
            nn.Linear(n_inputs, n_hidden1), nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2), nn.ReLU(),
            [YOUR CODE]  # the output layer: one score per class
        )

    def forward(self, X):
        [YOUR CODE]  # return ... (run the stack on X)

torch.manual_seed(42)
model = ImageClassifier(n_inputs=28 * 28, n_hidden1=300, n_hidden2=100, n_classes=10).to(device)
print("parameters:", sum(p.numel() for p in model.parameters()))   # should print 266610

**3c.** Choose the loss for classification into 10 classes, and train for 2 epochs with your `train()` from exercise 1.

In [ ]:
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=256)
[YOUR CODE]  # xentropy = ... (the loss for multiclass classification)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
train(model, optimizer, xentropy, train_loader, n_epochs=2)

**3d.** Measure the validation accuracy with `torchmetrics.Accuracy`. You should get about 85%.

In [ ]:
[YOUR CODE]  # accuracy = ... (a TorchMetrics accuracy for 10 classes, on device)
model.eval()
with torch.no_grad():
    for X_batch, y_batch in valid_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        accuracy.update(model(X_batch), y_batch)
print("validation accuracy:", accuracy.compute().item())

**3e.** Look at some predictions. Which mistakes does the model make? Do they make sense?

In [ ]:
X_new, y_new = next(iter(valid_loader))
X_new, y_new = X_new[:8].to(device), y_new[:8]
with torch.no_grad():
    logits = model(X_new)
y_proba = torch.softmax(logits, dim=1)   # probabilities, only to show them
y_pred = logits.argmax(dim=1).cpu()      # the class: argmax works on the logits directly

fig, axes = plt.subplots(1, 8, figsize=(14, 2.4))
for ax, image, true, pred, p in zip(axes, X_new.cpu(), y_new, y_pred, y_proba.max(dim=1).values.cpu()):
    ax.imshow(image.squeeze(), cmap="binary")
    ax.set_title(f"{class_names[pred]}\n({p:.0%})", fontsize=9, color="green" if pred == true else "red")
    ax.axis("off")
plt.show()

---
## Exercise 4: Save, load, and check (7 min)

**4a.** Save only the learned weights: the model's `state_dict()`.

In [ ]:
[YOUR CODE]  # save the weights

**4b.** Build a new, untrained model with the same shape, and load the weights into it.

In [ ]:
new_model = ImageClassifier(n_inputs=28 * 28, n_hidden1=300, n_hidden2=100, n_classes=10).to(device)
[YOUR CODE]  # load the weights
new_model.eval()

**4c.** Check that the two models give exactly the same outputs.

In [ ]:
with torch.no_grad():
    [YOUR CODE]  # same = ... (True if the outputs are exactly equal)
print("same outputs:", same)                                       # should print True

**Question:** why do we save the `state_dict()`, and not the whole model object?
*(Write your answer here.)*

**4d.** Keep your file. Files in Colab are deleted when the runtime ends. Either download it (Files panel on the left,
or run the first two lines below in a new cell), or save it to your Google Drive (the last three lines):

```python
from google.colab import files
files.download("my_fashion_model.pt")

from google.colab import drive
drive.mount("/content/drive")
torch.save(model.state_dict(), "/content/drive/MyDrive/my_fashion_model.pt")
```

---
## Exit questions (2 min)

Answer briefly, here or on paper:

1. Why does the classifier end without a softmax?
2. Why is the average of the batch RMSEs not the RMSE of the whole validation set?
3. What was the most confusing part of today?